In [ ]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Pytorch dataset
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
def load_and_tokenisation(dataset_path):
  # Load and Prepare Data
  print("\n" + "=" * 60)
  print("DATA LOADING")
  print("=" * 60)

  df = pd.read_csv(dataset_path).dropna()
  print(f"✓ Dataset loaded: {len(df)} rows")
  print(f"✓ Columns: {df.columns.tolist()}")
  print(f"\nLabel distribution:")
  print(df['label'].value_counts())

  # Train/Validation/Test Split
  print("\n" + "=" * 60)
  print("DATA SPLITTING")
  print("=" * 60)

  # First split: 70% train, 30% temp (later will be val + test)
  train_texts, temp_texts, train_labels, temp_labels = train_test_split(
      df['combined_text'].tolist(),
      df['label'].tolist(),
      test_size=0.3, # Changed to 0.3 to get 70% train
      random_state=42,
      stratify=df['label']
  )

  # Temp split: 50% validation, 50% test from the 30% temp data (which means 15% val, 15% test from total set)
  val_texts, test_texts, val_labels, test_labels = train_test_split(
      temp_texts,
      temp_labels,
      test_size=0.5,
      random_state=42,
      stratify=temp_labels
  )

  print(f"✓ Training samples: {len(train_texts)}")
  print(f"✓ Validation samples: {len(val_texts)}")
  print(f"✓ Test samples: {len(test_texts)}")

  # BERT Tokenization
  print("\n" + "=" * 60)
  print("TOKENIZATION")
  print("=" * 60)

  tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  print("✓ Tokenizer loaded")

  print("Tokenizing training data...")
  train_encodings = tokenizer(
      train_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Training data tokenized")

  print("Tokenizing validation data...")
  val_encodings = tokenizer(
      val_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Validation data tokenized")

  print("Tokenizing test data...")
  test_encodings = tokenizer(
      test_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Test data tokenized")

  #Create PyTorch Dataset
  print("\n" + "=" * 60)
  print("DATASET CREATION")
  print("=" * 60)

  train_dataset = FakeNewsDataset(train_encodings, train_labels)
  val_dataset = FakeNewsDataset(val_encodings, val_labels)
  test_dataset = FakeNewsDataset(test_encodings, test_labels)

  print(f"✓ Train dataset: {len(train_dataset)} samples")
  print(f"✓ Validation dataset: {len(val_dataset)} samples")
  print(f"✓ Test dataset: {len(test_dataset)} samples")

  return train_dataset, val_dataset, test_dataset, tokenizer

In [ ]:
def load_model():
  # setup device

  # Primarily using Colab's TPU, on occasion will use Colab's GPU runtimes for the neural networks and machine learning tasks but primarily using TPU for BERT
  try:
      import torch_xla.core.xla_model as xm
      device = xm.xla_device()
      print(f"✓ Using TPU: {device}")
      use_tpu = True
  except ImportError:
      use_tpu = False
      if torch.cuda.is_available():
          device = torch.device("cuda")
          print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
          print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
      else:
          device = torch.device("cpu")
          print("⚠ Using CPU (Training will be slow!)")

  # Load Pre-trained BERT Model
  print("\n" + "=" * 60)
  print("MODEL LOADING")
  print("=" * 60)

  model = BertForSequenceClassification.from_pretrained(
      'bert-base-uncased',
      num_labels=2
  )

  if not use_tpu:
      model.to(device)
      print(f"✓ BERT model loaded and moved to {device}")
  else:
      print("✓ BERT model loaded (TPU will handle device placement)")

  return model, device, use_tpu

In [ ]:
def compute_metrics(pred):
    """Compute accuracy, precision, recall, F1"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [ ]:
# def finetuning(model, train_dataset, val_dataset, test_dataset, compute_metrics_fn, use_tpu, device):
#   print("\n" + "=" * 60)
#   print("TRAINING CONFIGURATION")
#   print("=" * 60)

#   # 8. Define Training Arguments
#   training_args = TrainingArguments(
#       output_dir='./results',
#       num_train_epochs=3,
#       per_device_train_batch_size=32,
#       per_device_eval_batch_size=64,
#       warmup_steps=500,
#       weight_decay=0.01,
#       logging_dir='./logs',
#       logging_steps=50,
#       eval_strategy="steps",
#       eval_steps=1000,
#       save_strategy="steps",
#       save_steps=1000,
#       save_total_limit=2,
#       load_best_model_at_end=True,
#       metric_for_best_model="f1",
#       greater_is_better=True,
#       optim="adamw_torch",
#       fp16=False,
#       dataloader_num_workers=0,
#       report_to="none",
#   )

#   print("Training configuration:")
#   print(f"  - Epochs: {training_args.num_train_epochs}")
#   print(f"  - Train batch size: {training_args.per_device_train_batch_size}")
#   print(f"  - Eval batch size: {training_args.per_device_eval_batch_size}")
#   print(f"  - FP16: {training_args.fp16}")
#   print(f"  - Device: {'TPU' if use_tpu else device}")


#   print("\n" + "=" * 60)
#   print("TRAINER INITIALIZATION")
#   print("=" * 60)

#   trainer = Trainer(
#       model=model,
#       args=training_args,
#       train_dataset=train_dataset,
#       eval_dataset=val_dataset, # The eval_dataset here is for validation during training
#       compute_metrics=compute_metrics_fn
#   )
#   print("✓ Trainer initialized")

#   print("\n" + "=" * 60)
#   print("TRAINING START")
#   print("=" * 60)

#   trainer.train()
#   print("\n✓ Training completed!")

#   print("\n" + "=" * 60)
#   print("VALIDATION SET EVALUATION")
#   print("=" * 60)

#   val_results = trainer.evaluate()
#   print("Validation Results:")
#   for key, value in val_results.items():
#       print(f"  {key}: {value:.4f}")

#   print("\n" + "=" * 60)
#   print("TEST SET EVALUATION")
#   print("=" * 60)

#   test_results = trainer.evaluate(eval_dataset=test_dataset)
#   print("Test Results:")
#   for key, value in test_results.items():
#       print(f"  {key}: {value:.4f}")

In [ ]:
def cross_validate_finetuning(full_train_dataset, compute_metrics_fn, use_tpu, device, n_splits=5):
    print("\n" + "=" * 60)
    print(f"STARTING {n_splits}-FOLD CROSS VALIDATION (Pre-tokenized Data)")
    print("=" * 60)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    # Get labels from the dataset object to perform stratified split
    labels = full_train_dataset.labels
    indices = np.arange(len(labels))

    fold_accuracies = []
    fold_f1_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        print(f"\n--- FOLD {fold + 1} ---")

        # Create subsets using indices
        train_subset = Subset(full_train_dataset, train_idx)
        val_subset = Subset(full_train_dataset, val_idx)

        # Load fresh model instance
        model, _, _ = load_model()

        training_args = TrainingArguments(
            output_dir=f'./results_fold_{fold+1}',
            num_train_epochs=3,
            per_device_train_batch_size=128,
            per_device_eval_batch_size=128,
            eval_strategy="steps",
            eval_steps=512,
            save_strategy="steps",
            save_steps=512,
            save_total_limit=2,
            report_to="none",
            optim="adamw_torch",
            logging_dir='./logs',
            logging_steps=512,
            metric_for_best_model="f1",
            load_best_model_at_end=True,
            weight_decay=0.01,
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_subset,
            eval_dataset=val_subset,
            compute_metrics=compute_metrics_fn
        )

        trainer.train()
        eval_metrics = trainer.evaluate()

        fold_accuracies.append(eval_metrics['eval_accuracy'])
        fold_f1_scores.append(eval_metrics['eval_f1'])
        print(f"Fold {fold+1} Accuracy: {eval_metrics['eval_accuracy']:.4f}, F1-score: {eval_metrics['eval_f1']:.4f}")

    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    for i in range(n_splits):
        print(f"Fold {i+1} - Accuracy: {fold_accuracies[i]:.4f}, F1-score: {fold_f1_scores[i]:.4f}")

    avg_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    avg_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)

    print(f"\nAverage Accuracy: {avg_acc:.4f} (Std Dev: {std_acc:.4f})")
    print(f"Average F1-score: {avg_f1:.4f} (Std Dev: {std_f1:.4f})")

    return model

In [ ]:
def save_model(model, tokenizer, output_path, use_tpu, device):
  print("\n" + "=" * 60)
  print("MODEL SAVING")
  print("=" * 60)

  output_dir = output_path

  # Move model to CPU before saving if it's on TPU
  if use_tpu:
      model.to("cpu")
      print("Moved model to CPU for saving.")

  model.save_pretrained(output_dir)
  tokenizer.save_pretrained(output_dir)
  print(f"✓ Model saved to {output_dir}")

  # Move model back to original device after saving
  if use_tpu:
      model.to(device)
      print("Moved model back to TPU.")

  print("\n" + "=" * 60)
  print("BERT FINE-TUNING COMPLETE! 🎉")
  print("=" * 60)

In [ ]:
def cross_dataset_evaluation(model, tokenizer, current_dataset_name, all_datasets_paths, compute_metrics_fn):
    print("\n" + "!" * 60)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 60)

    results = {}

    for name, path in all_datasets_paths.items():
        if name == current_dataset_name:
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        # Load full unseen dataset for testing
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
        dataset = FakeNewsDataset(encodings, test_labels)

        # Trainer for evaluation
        eval_trainer = Trainer(
            model=model,
            compute_metrics=compute_metrics_fn
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics
        print(f"  -> {name} Accuracy: {metrics['eval_accuracy']:.4f}, F1: {metrics['eval_f1']:.4f}")

    return results

In [ ]:
# Dictionary of all 5 datasets
DATASETS = {
    "WELFake": "/content/drive/MyDrive/datasets/WELFake_processed.csv",
    "FakeNewsNet": "/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv",
    "Fake_News_Detection": "/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv",
    "ISOT": "/content/drive/MyDrive/datasets/ISOT_processed.csv",
    "Fake_News_Classification": "/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv"
}

def new_train_loop(dataset_name, output_path, n_splits=5):
    dataset_path = DATASETS[dataset_name]

    # Data Loading & Tokenization
    print(f"\nInitialising experiment on: {dataset_name}")
    df = pd.read_csv(dataset_path).dropna()
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    print(f"Tokenizing {dataset_name} for Cross-Validation...")
    encodings = tokenizer(
        df['combined_text'].tolist(),
        truncation=True,
        padding=True,
        max_length=128
    )
    full_dataset = FakeNewsDataset(encodings, df['label'].tolist())

    # 5-Fold Cross Validation
    model, device, use_tpu = load_model()
    trained_model = cross_validate_finetuning(
        full_dataset,
        compute_metrics,
        use_tpu,
        device,
        n_splits=n_splits
    )

    # Save Model
    save_model(trained_model, tokenizer, output_path, use_tpu, device)

    # Generalization Testing (Against all other datasets)
    cross_dataset_evaluation(trained_model, tokenizer, dataset_name, DATASETS, compute_metrics)

In [ ]:
# def train_loop(dataset_path, output_path):
#   train_dataset, val_dataset, test_dataset, tokenizer = load_and_tokenisation(dataset_path)
#   model, device, use_tpu = load_model()
#   finetuning(model, train_dataset, val_dataset, test_dataset, compute_metrics, use_tpu, device)
#   save_model(model, tokenizer, output_path, use_tpu, device)

# WELFake Dataset

In [ ]:
new_train_loop("WELFake", "/content/drive/MyDrive/bert_models/WELFake")


Initialising experiment on: WELFake


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> Fake_News_Detection Accuracy: 0.2098, F1: 0.3464

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9995, F1: 0.9994

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0185, F1: 0.0355


In [ ]:
# train_loop("/content/drive/MyDrive/datasets/WELFake_processed.csv", "/content/drive/MyDrive/bert_models/WELFake")


DATA LOADING
✓ Dataset loaded: 63670 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
0    34790
1    28880
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 44569
✓ Validation samples: 9550
✓ Test samples: 9551

TOKENIZATION


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0423
  eval_accuracy: 0.9923
  eval_f1: 0.9915
  eval_precision: 0.9879
  eval_recall: 0.9952
  eval_runtime: 4.9289
  eval_samples_per_second: 973.8390
  eval_steps_per_second: 15.2160
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0543
  eval_accuracy: 0.9898
  eval_f1: 0.9889
  eval_precision: 0.9844
  eval_recall: 0.9933
  eval_runtime: 12.7785
  eval_samples_per_second: 375.6310
  eval_steps_per_second: 5.8690
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/WELFake
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# FakeNewsNet

In [ ]:
new_train_loop("FakeNewsNet", "/content/drive/MyDrive/bert_models/FakeNewsNet")


Initialising experiment on: FakeNewsNet


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> Fake_News_Detection Accuracy: 0.5084, F1: 0.4356

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.6021, F1: 0.4042

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.3996, F1: 0.2026


In [ ]:
# train_loop("/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv", "/content/drive/MyDrive/bert_models/FakeNewsNet")


DATA LOADING
✓ Dataset loaded: 21844 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
1    16522
0     5322
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 15290
✓ Validation samples: 3277
✓ Test samples: 3277

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 15290 samples
✓ Validation dataset: 3277 samples
✓ Test dataset: 3277 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING
✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.193500,0.472684,0.841013,0.896029,0.886651,0.905607



✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.4727
  eval_accuracy: 0.8410
  eval_f1: 0.8960
  eval_precision: 0.8867
  eval_recall: 0.9056
  eval_runtime: 1.3920
  eval_samples_per_second: 1195.3660
  eval_steps_per_second: 18.6780
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.4635
  eval_accuracy: 0.8499
  eval_f1: 0.9016
  eval_precision: 0.8934
  eval_recall: 0.9100
  eval_runtime: 18.5914
  eval_samples_per_second: 89.5040
  eval_steps_per_second: 1.3980
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/FakeNewsNet
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# Fake News Detection Dataset

In [ ]:
new_train_loop("Fake_News_Detection", "/content/drive/MyDrive/bert_models/Fake_News_Detection")


Initialising experiment on: Fake_News_Detection


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> FakeNewsNet Accuracy: 0.4574, F1: 0.5060

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.0007, F1: 0.0012

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.9819, F1: 0.9830


In [ ]:
# train_loop("/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv", "/content/drive/MyDrive/bert_models/Fake_News_Detection")


DATA LOADING
✓ Dataset loaded: 38650 rows
✓ Columns: ['combined_text', 'label']

Label distribution:
label
1    21194
0    17456
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 27055
✓ Validation samples: 5797
✓ Test samples: 5798

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 27055 samples
✓ Validation dataset: 5797 samples
✓ Test dataset: 5798 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.016600,0.013096,0.996032,0.996379,0.997478,0.995282
2000,0.000100,0.009315,0.997585,0.997801,0.996548,0.999056


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0093
  eval_accuracy: 0.9976
  eval_f1: 0.9978
  eval_precision: 0.9965
  eval_recall: 0.9991
  eval_runtime: 3.0752
  eval_samples_per_second: 946.9270
  eval_steps_per_second: 14.9580
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0128
  eval_accuracy: 0.9969
  eval_f1: 0.9972
  eval_precision: 0.9950
  eval_recall: 0.9994
  eval_runtime: 10.2519
  eval_samples_per_second: 284.0440
  eval_steps_per_second: 4.4870
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/Fake_News_Detection
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# ISOT Dataset

In [ ]:
new_train_loop("ISOT", "/content/drive/MyDrive/bert_models/ISOT")


Initialising experiment on: ISOT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> FakeNewsNet Accuracy: 0.7522, F1: 0.8583

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.5084, F1: 0.6741

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0183, F1: 0.0359


In [ ]:
#train_loop("/content/drive/MyDrive/datasets/ISOT_processed.csv", "/content/drive/MyDrive/bert_models/ISOT")

# Fake News Classification Dataset

In [ ]:
# train_loop("/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv", "/content/drive/MyDrive/bert_models/Fake_News_Detection")

In [ ]:
new_train_loop("Fake_News_Classification", "/content/drive/MyDrive/bert_models/Fake_News_Classification")


Initialising experiment on: Fake_News_Classification
Tokenizing Fake_News_Classification for Cross-Validation...


/tmp/ipykernel_9877/197324397.py:7: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0

MODEL LOADING


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_9877/197324397.py:7: D

...

  -> FakeNewsNet Accuracy: 0.3571, F1: 0.3282

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.4889, F1: 0.1272

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.0006, F1: 0.0006
